# subfloor — bulutta bir nokta

Bu defter **ana projeye dokunmuyor**: koşan şey `experiments/m1_run.py`, yerelde
koşanın aynısı. `cloud/` yalnız ücretsiz bir oturumun ihtiyacı olanı ekliyor —
duvar saati bütçesi ve varsayılan devam etme.

**3. hücreyi atlamayın.** Bu projedeki her zamanlama sabiti "bu makinede ölçüldü"
diyor ve bu makine o makine değil. Ayrıntı: `cloud/README.md`.


## 1 — Kodu getir

Üç yol, sırayla denenir:

1. **Zip** — depoda git remote yoksa bu. Yerelde `git archive --format=zip
   -o subfloor-code.zip HEAD` ile üretilir (0.4 MB). Kaggle'da Dataset olarak
   ekleyin, Colab'da Drive'a koyun; defter kendisi bulur.
2. **`REPO_URL`** — uzak depo varsa doldurun.
3. Kod zaten açılmışsa hiçbir şey yapmaz.

Model **NousResearch aynası**: kapılı değil, HuggingFace token'ı gerekmiyor.


In [ ]:
REPO_URL = ''        # uzak depo varsa: 'https://github.com/KULLANICI/subfloor'
CODE_ZIP = ''        # ya da zip'in tam yolu; bos birakirsaniz aranir

import os, sys, glob, zipfile, subprocess, shutil, pathlib

if pathlib.Path('/kaggle').exists():
    PLATFORM, ROOT = 'kaggle', pathlib.Path('/kaggle/working')
elif pathlib.Path('/content').exists():
    PLATFORM, ROOT = 'colab', pathlib.Path('/content')
else:
    PLATFORM, ROOT = 'local', pathlib.Path.cwd()
REPO = ROOT / 'subfloor'

def _find_zip():
    if CODE_ZIP and pathlib.Path(CODE_ZIP).exists():
        return CODE_ZIP
    for pat in ('/kaggle/input/**/subfloor-code.zip',
                '/content/drive/MyDrive/**/subfloor-code.zip',
                str(ROOT / '**' / 'subfloor-code.zip')):
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None

if (REPO / 'quantize.py').exists():
    print('kod zaten burada:', REPO)
elif (z := _find_zip()):
    REPO.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(z).extractall(REPO)
    print('acildi:', z, '->', REPO)
elif REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    raise SystemExit('kod yok: subfloor-code.zip yukleyin ya da REPO_URL doldurun')

os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'cloud/requirements.txt'], check=True)

import torch
p = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print(PLATFORM, '|', p.name if p else 'GPU YOK -- calisma zamanini degistirin',
      f'{p.total_memory / 2**30:.1f} GiB' if p else '')


## 2 — Checkpoint nereye gidecek

**Bir nokta ~13 GiB tutuyor** ve 32 blok dosyasının hepsi sonuna kadar gerekli:
değerlendirme birleştirilmiş sıkıştırılmış model üzerinde koşuyor. Disk yetmezse
koşu *geç* çuvallar.

- **Kaggle**: `/kaggle/working` (20 GB) sığar. Oturumlar arası kalıcılık için
  notebook'u *Save Version* ile koşturun.
- **Colab**: ücretsiz Drive 15 GB, modelle birlikte **yetmez**. Ucuz noktaları
  (T=8, 16, 32, max) yerel diske yazıp tek oturumda bitirin.


In [ ]:
USE_DRIVE = False        # Colab'da True: Drive baglanir (yavas ama kalici)

if PLATFORM == 'colab' and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESUME = pathlib.Path('/content/drive/MyDrive/subfloor/resume')
else:
    RESUME = ROOT / 'resume'
RESUME.mkdir(parents=True, exist_ok=True)
print(f'{RESUME}  {shutil.disk_usage(RESUME).free / 2**30:.1f} GiB bos  (bir nokta ~13 istiyor)')


## 3 — Uçuş öncesi

Ortamı, diski ve hattın kendisini sınar; sonra **çekirdek hızlarını bu makinede**
yeniden ölçer. Sonunda, yeniden ölçemediği ve yeni bir kartın geçersiz
kılabileceği beş eşiği basar — `docs/STATUS.md` §6.13 o hatanın ne kadara mal
olduğunu anlatıyor.


In [ ]:
!python cloud/preflight.py --resume-root {RESUME}


## 4 — Bir nokta koştur

Tasarım F'in yedi noktası **bağımsız**. Modelin dediği süreler (yerel karta göre;
bulutta yeniden ölçülecek): T=1 4.01s, T=2 3.34, T=4 2.07, T=8 1.54, T=16 1.14,
T=32 0.92, T=max 0.66.

Bütçe dolarsa süreç **42** ile çıkar ve checkpoint tamdır — aynı hücreyi tekrar
koşturun, kaldığı yerden devam eder.

`--calib-seqlen 2048`, `m1_run.py`'nin 4096 varsayılanı **değil**: C4 belgelerinin
yalnız %0.33'ü 4096 token'ı aşıyor ve örnekleyicinin deneme bütçesi yetişmiyor
(`cloud/README.md`).


In [ ]:
TILE  = '16'      # 1, 2, 4, 8, 16, 32, max
HOURS = 11.0 if PLATFORM == 'kaggle' else 3.5

cmd = [sys.executable, '-u', 'cloud/run_point.py',
       '--tile', TILE, '--budget', '1.5', '--draw', '0',
       '--resume-root', str(RESUME), '--hours', str(HOURS),
       '--calib-samples', '128', '--calib-seqlen', '2048',
       '--datasets', 'wikitext2']
rc = subprocess.run(cmd).returncode
print({0: 'BITTI', 42: 'BUTCE DOLDU -- bu hucreyi tekrar kosturun'}
      .get(rc, f'HATA (exit {rc})'))


## 5 — Sonucu al

Bitmiş bir nokta `<resume-root>/<slug>.json` dosyasına yazılıyor: perplexity,
katman başına göreli hata ve SNR, blok 0'ın yoğun E8P referansı (§3.2'nin
erken-uyarı kuralı), ve koşunun hangi kaldıraçlarla alındığı.


In [ ]:
import json, glob
for f in sorted(glob.glob(str(RESUME / '*.json'))):
    d = json.load(open(f))
    if 'perplexity' not in d: continue
    print(f"T={d['spec']['tile_size']:<4} {d['seconds']/3600:5.2f} h  "
          f"{d['perplexity']}  {d['levers']}")

if PLATFORM == 'colab' and pathlib.Path('/content/drive').exists():
    !mkdir -p /content/drive/MyDrive/subfloor && cp {RESUME}/*.json /content/drive/MyDrive/subfloor/
